# SIXray-D: feature KD



In [ ]:
from pathlib import Path
import sys
import json
import random
import hashlib
from datetime import datetime, timezone
from importlib.metadata import version

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path[:0] = [str(REPO_ROOT), str(REPO_ROOT / 'students/anna_student_resnet18/src')]

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from src.utils.checkpoint import load_checkpoint
from src.engine.inference import prepare_validation_files
from sixray_student.model import build_student_model
from sixray_student.config import CLASS_NAMES, GRID_SIZE, NUM_BOXES, BBOX_ENCODING
from sixray_student.data import SixRayStudentDataset, detection_collate_fn, load_split
from sixray_student.losses import yolo_loss, feature_distillation_loss
from sixray_student.target_encoder import encode_targets_yolo
from sixray_student.metrics import evaluate_map
from sixray_student.train_utils import train_one_epoch, make_grad_scaler


## settings



In [ ]:
ARTIFACT_ROOT = Path(r'C:\Users\89526\Documents\sixray-artifacts')
PROJECT_DIR = ARTIFACT_ROOT / 'DatasetAPAI/SIXray_Project'
DATA_DIR = ARTIFACT_ROOT / 'DatasetAPAI/subset_clean'
ARCHIVE_PATH = ARTIFACT_ROOT / 'DatasetAPAI/subset_clean.zip'
SPLIT_PATH = PROJECT_DIR / 'splits/split_seed42_train10500_val300pos1200neg.json'
TEACHER_DIR = PROJECT_DIR / 'checkpoints_angelo/02_rtdetr_teacher_oversampling_angelo_best'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CONFIG = {
    'seed': 42, 'adapter_seed': 43, 'epochs': 50, 'batch_size': 2, 'num_workers': 2,
    'learning_rate': 1e-4, 'weight_decay': 1e-4, 'eta_min': 1e-6,
    'grad_clip_norm': 1.0, 'augmentation': 'resize_only', 'image_size': 640,
    'kd_lambda': 1.0, 'background_weight': 0.1, 'selection_metric': 'map',
    'score_threshold': 0.001, 'nms_iou': 0.5, 'max_detections': 100,
    'evaluation_frame': 'original_image_xyxy', 'evaluation_amp': False,
    'training_amp': DEVICE.type == 'cuda',
}

def seed_run(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [ ]:
train_indices, val_indices, _ = load_split(SPLIT_PATH)
train_data = SixRayStudentDataset(DATA_DIR / 'train/images', DATA_DIR / 'train.json', train_indices)
val_data = SixRayStudentDataset(DATA_DIR / 'train/images', DATA_DIR / 'train.json', val_indices)

EXPERIMENT_DIR = ARTIFACT_ROOT / 'experiments' / (
    'feature_kd_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
)
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=False)
seed_run(CONFIG['seed'])
import shutil
INITIAL_PATH = EXPERIMENT_DIR / 'initial_student.pth'
shutil.copy2(ARTIFACT_ROOT / 'experiments/feature_kd_20261016T214646677055Z/initial_student.pth', INITIAL_PATH)

processor, teacher, teacher_meta = load_checkpoint(TEACHER_DIR, device=DEVICE, strict=True)
teacher.requires_grad_(False).eval()


## Reproducibility and separate outputs

The initial student state is shared by both runs. Checkpoints store only student parameters under `model_state_dict`; adapter weights have a separate key and are never needed for inference.

In [ ]:
metadata = {
    **CONFIG, 'device': str(DEVICE), 'split_path': str(SPLIT_PATH),
    'split_sha256': hashlib.sha256(SPLIT_PATH.read_bytes()).hexdigest(),
    'initial_state_sha256': hashlib.sha256(INITIAL_PATH.read_bytes()).hexdigest(),
    'teacher_path': str(TEACHER_DIR),
    'teacher_sha256': hashlib.sha256((TEACHER_DIR / 'model.safetensors').read_bytes()).hexdigest(),
    'teacher_feature': 'model.backbone.model(...).feature_maps[-1]',
    'student_feature': 'backbone(normalizer(images))',
    'teacher_provenance': 'saved oversampling checkpoint; resume model restoration absent in notebook 04',
    'versions': {p: version(p) for p in ('torch', 'torchvision', 'transformers', 'torchmetrics', 'pycocotools')},
    'source_sha256': {name: hashlib.sha256((REPO_ROOT / name).read_bytes()).hexdigest() for name in (
        'students/anna_student_resnet18/src/sixray_student/model.py',
        'students/anna_student_resnet18/src/sixray_student/losses.py',
        'students/anna_student_resnet18/src/sixray_student/train_utils.py',
        'students/anna_student_resnet18/src/sixray_student/data.py',
        'students/anna_student_resnet18/src/sixray_student/metrics.py')},
}
for name, kd_lambda in [('control', 0.0), ('kd', CONFIG['kd_lambda'])]:
    run_dir = EXPERIMENT_DIR / name
    run_dir.mkdir()
    with (run_dir / 'config.json').open('x', encoding='utf-8') as output:
        json.dump({**metadata, 'run': name, 'kd_lambda': kd_lambda}, output, indent=2)
print(EXPERIMENT_DIR)


## Feature loss and box projection

The teacher receives the same resized `[0,1]` tensor as the student; only the student applies ImageNet normalization. We use teacher `[B,2048,20,20]` and student `[B,512,20,20]` backbone features, with a training-only `Conv2d(512,2048,1,bias=False)`.

For an original image of width W and height H, a box corner becomes `(x*640/W, y*640/H)` in the dataset and then `(x*20/W, y*20/H)` on the feature grid. Floor left/top and ceil right/bottom, covering every intersecting cell. Boxes overlap by taking their mask union.

Normalize channels after the adapter. At each cell, sum squared channel differences. Average foreground and background distances separately per image, then use `L_KD = mean(L_foreground + 0.1*L_background)`. Empty foreground contributes zero; negative images retain the background term. Total loss is the **unchanged** `L_objectness + 10*L_box + L_class`, plus `1.0*L_KD`.


In [ ]:
seed_run(CONFIG['seed'])
student = build_student_model(pretrained_backbone=False).to(DEVICE)
student.load_state_dict(torch.load(INITIAL_PATH, map_location=DEVICE, weights_only=True))
student.train()
with torch.random.fork_rng():
    torch.manual_seed(CONFIG['adapter_seed'])
    adapter = torch.nn.Conv2d(512, 2048, 1, bias=False).to(DEVICE)

positive = next(p for p, i in enumerate(train_indices)
                if train_data.annotations_by_image_id.get(train_data.images[i]['id']))
negative = next(p for p, i in enumerate(train_indices)
                if not train_data.annotations_by_image_id.get(train_data.images[i]['id']))
check_positions = [positive, negative]
prepare_validation_files(ARCHIVE_PATH, DATA_DIR, [train_indices[p] for p in check_positions])
images, targets = next(iter(DataLoader(Subset(train_data, check_positions), batch_size=2,
                                      collate_fn=detection_collate_fn, num_workers=0)))
images = images.to(DEVICE)
encoded = encode_targets_yolo(targets, device=DEVICE)
features = student.backbone(student.normalizer(images))
detection_loss, detection_values = yolo_loss(student.head(features), encoded)
with torch.no_grad():
    teacher_features = teacher.model.backbone.model(images).feature_maps[-1]
kd_loss, kd_values = feature_distillation_loss(
    features, teacher_features, adapter, targets, background_weight=CONFIG['background_weight'],
)


In [ ]:
def gradient_norm(gradients):
    return float(torch.stack([g.detach().float().square().sum() for g in gradients]).sum().sqrt())

backbone_parameters = tuple(student.backbone.parameters())
detection_gradients = torch.autograd.grad(detection_loss, backbone_parameters, retain_graph=True)
kd_gradients = torch.autograd.grad(kd_loss, backbone_parameters, retain_graph=True)
adapter_gradients = torch.autograd.grad(kd_loss, tuple(adapter.parameters()), retain_graph=True)
report = {
    'detection': detection_values, **kd_values,
    'weighted_bbox_loss': 10 * detection_values['bbox_loss'],
    'combined_loss': float((detection_loss + CONFIG['kd_lambda'] * kd_loss).detach()),
    'backbone_detection_gradient_norm': gradient_norm(detection_gradients),
    'backbone_kd_gradient_norm': gradient_norm(kd_gradients),
    'adapter_kd_gradient_norm': gradient_norm(adapter_gradients),
    'student_feature_shape': list(features.shape), 'teacher_feature_shape': list(teacher_features.shape),
    'image_ids': [int(t['image_id'].item()) for t in targets],
    'objects_per_image': [len(t['boxes']) for t in targets],
    'source_indices': [train_indices[p] for p in check_positions],
    'precision': 'float32', 'optimizer_steps': 0,
}
(detection_loss + CONFIG['kd_lambda'] * kd_loss).backward()
report['backbone_combined_gradient_norm'] = gradient_norm(p.grad for p in backbone_parameters)
report['adapter_combined_gradient_norm'] = gradient_norm(p.grad for p in adapter.parameters())
report['teacher_trainable_parameters'] = sum(p.numel() for p in teacher.parameters() if p.requires_grad)
report['teacher_parameters_with_gradients'] = sum(p.grad is not None for p in teacher.parameters())
report['teacher_feature_requires_grad'] = teacher_features.requires_grad
print(json.dumps(report, indent=2))
with (EXPERIMENT_DIR / 'single_batch.json').open('x', encoding='utf-8') as output:
    json.dump(report, output, indent=2, allow_nan=False)

del student, adapter, features, teacher_features, detection_loss, kd_loss


## Matched training runs — wait for approval

Each run reloads the same initial state and resets the model/data seeds. A dedicated DataLoader generator keeps sample ordering independent of adapter creation. Both use identical AMP training settings and FP32 validation. No test images are used. Best checkpoints are selected by full-validation COCO mAP; AP50, AP75 and per-class AP are retained.

The existing epoch cosine advances only after 5,250 successful optimizer updates. AMP skips do not advance it; skipped updates delay the next scheduler boundary.


In [ ]:
def run_experiment(name):
    seed_run(CONFIG['seed'])
    model = build_student_model(pretrained_backbone=False).to(DEVICE)
    model.load_state_dict(torch.load(INITIAL_PATH, map_location=DEVICE, weights_only=True))
    kd_adapter = None
    parameters = list(model.parameters())
    if name == 'kd':
        with torch.random.fork_rng():
            torch.manual_seed(CONFIG['adapter_seed'])
            kd_adapter = torch.nn.Conv2d(512, 2048, 1, bias=False).to(DEVICE)
        parameters += list(kd_adapter.parameters())
    generator = torch.Generator().manual_seed(CONFIG['seed'])
    train_loader = DataLoader(train_data, batch_size=CONFIG['batch_size'], shuffle=True,
                              generator=generator, num_workers=CONFIG['num_workers'],
                              collate_fn=detection_collate_fn)
    val_loader = DataLoader(val_data, batch_size=CONFIG['batch_size'], shuffle=False,
                            num_workers=CONFIG['num_workers'], collate_fn=detection_collate_fn,
                            generator=torch.Generator().manual_seed(CONFIG['seed']))
    optimizer = torch.optim.AdamW(parameters, lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'], eta_min=CONFIG['eta_min'])
    scaler = make_grad_scaler(DEVICE, use_amp=CONFIG['training_amp'])
    history, best_map = [], -1.0
    run_dir = EXPERIMENT_DIR / name
    for epoch in range(1, CONFIG['epochs'] + 1):
        train_metrics = train_one_epoch(
            model, train_loader, optimizer, DEVICE, epoch, scaler,
            use_amp=CONFIG['training_amp'], grad_clip_norm=CONFIG['grad_clip_norm'],
            teacher=teacher if name == 'kd' else None, adapter=kd_adapter,
            kd_weight=CONFIG['kd_lambda'], background_weight=CONFIG['background_weight'], scheduler=scheduler,
        )
        validation = evaluate_map(model, val_loader, DEVICE, confidence_threshold=CONFIG['score_threshold'],
                                  nms_iou_threshold=CONFIG['nms_iou'], max_detections=CONFIG['max_detections'],
                                  use_amp=False, original_coordinates=True)
        history.append({'epoch': epoch, 'train': train_metrics, 'validation': validation})
        checkpoint = {
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'adapter_state_dict': kd_adapter.state_dict() if kd_adapter is not None else None,
            'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(), 'data_generator_state': generator.get_state(),
            'torch_rng_state': torch.get_rng_state(), 'python_rng_state': random.getstate(),
            'numpy_rng_state': np.random.get_state(),
            'cuda_rng_states': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            'best_checkpoint_metric': 'map', 'validation': validation, 'history': history,
            'config': {**metadata, 'run': name, 'kd_lambda': CONFIG['kd_lambda'] if name == 'kd' else 0.0},
            'class_names': CLASS_NAMES, 'image_size': 640, 'grid_size': GRID_SIZE,
            'num_classes': len(CLASS_NAMES), 'num_boxes': NUM_BOXES, 'bbox_encoding': BBOX_ENCODING,
        }
        improved = validation['map'] > best_map
        best_map = max(best_map, validation['map'])
        checkpoint['best_val_map'] = best_map
        if improved:
            torch.save(checkpoint, run_dir / 'student_best_map.pth')
        torch.save(checkpoint, run_dir / 'student_last.pth')
        with (run_dir / 'history.json').open('w', encoding='utf-8') as output:
            json.dump(history, output, indent=2)
        print(f"{name} epoch {epoch}: mAP={validation['map']:.4f}, AP50={validation['map_50']:.4f}, AP75={validation['map_75']:.4f}")


In [ ]:
RUN_TRAINING = False  
if RUN_TRAINING:
    prepare_validation_files(ARCHIVE_PATH, DATA_DIR, train_indices + val_indices)
    run_experiment('control')
    run_experiment('kd')
